# SPHEREx SkyBlink — MVP
Run this notebook in Google Colab. It validates the detection core with a synthetic moving source and sketches the live IRSA SIA v2 path.

In [ ]:
!pip -q install numpy pandas matplotlib scipy astropy astroquery pyvo requests pillow

In [ ]:
import sys
sys.path.append('/content/spherex-skyblink')
import numpy as np
import matplotlib.pyplot as plt
from backend.science import detect_moving_candidates

In [ ]:
def gaussian(shape,x,y,amp=4,sigma=1.2):
    yy,xx=np.indices(shape)
    return amp*np.exp(-((xx-x)**2+(yy-y)**2)/(2*sigma**2))
rng=np.random.default_rng(123)
a=rng.normal(0,0.04,(128,128))
b=rng.normal(0,0.04,(128,128))
for x,y in [(30,30),(90,72),(48,106)]:
    a+=gaussian(a.shape,x,y,4)
    b+=gaussian(b.shape,x,y,4)
a+=gaussian(a.shape,60,80,5)
b+=gaussian(b.shape,65,76,5)
plt.figure(figsize=(6,6));plt.imshow(a,origin='lower',cmap='magma');plt.title('Epoch A');plt.show()
plt.figure(figsize=(6,6));plt.imshow(b,origin='lower',cmap='magma');plt.title('Epoch B');plt.show()

In [ ]:
candidates=detect_moving_candidates(a,b,significance_threshold=4,min_motion_px=.5)
for c in candidates[:10]: print(c)

## Live SPHEREx observation discovery via IRSA SIA v2
Keep the search radius small while prototyping. IRSA returns observation metadata including access_url.

In [ ]:
from pyvo.dal import sia2
from astropy.coordinates import SkyCoord
import astropy.units as u
service=sia2.SIAService('https://irsa.ipac.caltech.edu/SIA')
collection='spherex_qr3'
pos=SkyCoord(ra=83.633*u.deg,dec=-5.391*u.deg,frame='icrs')
# results=service.search(pos=(pos.ra.deg,pos.dec.deg,0.1*u.deg),collection=collection)
# table=results.to_table()
# print(table[:5])

## Production pipeline
1. Select compatible observations.
2. Request small cutouts.
3. Read MEF/WCS and propagate flags.
4. Reproject and subtract.
5. Link residuals across epochs.
6. Cross-match and human-review candidates.